In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.neighbors import KernelDensity # alternative or specialized libraries can be used, but manual NW or standard smoothers are direct
# For Nadaraya-Watson and Smoothing Splines, standard scientific tools or manual implementations:
from scipy.interpolate import UnivariateSpline

# Ensure reproducibility
np.random.seed(42)

# =====================================================================
# TASK 1: Nadaraya-Watson & Smoothing Splines Regression (10 points)
# =====================================================================
print("--- Running Task 1 ---")

# 1.1 Read the paired data (Simulating weights.csv for demonstration purposes)
# In your real run, replace this generation with: df = pd.read_csv('weights.csv')
try:
    df = pd.read_csv('weights.csv')
    x = df.iloc[:, 0].values
    y = df.iloc[:, 1].values
except FileNotFoundError:
    print("weights.csv not found. Generating dummy data for Task 1 demonstration.")
    x = np.sort(np.random.uniform(0, 10, 100))
    y = 70 + 5 * np.sin(x) + np.random.normal(0, 1, 100)

n = len(x)

# 1.2 Randomly split into 4 near equal-sized folds
kf = KFold(n_splits=4, shuffle=True, random_state=42)

# Define Nadaraya-Watson Kernel Regression manually for flexibility with bandwidth h
def nadaraya_watson(x_train, y_train, x_test, h):
    # Using Gaussian Kernel
    dists = (x_test[:, None] - x_train[None, :]) / h
    weights = np.exp(-0.5 * dists**2)
    sum_weights = np.sum(weights, axis=1)
    # Avoid division by zero for points far away
    sum_weights[sum_weights == 0] = 1e-10
    pred = np.sum(weights * y_train, axis=1) / sum_weights
    return pred

# --- 4-fold CV for Nadaraya-Watson ---
bandwidths = np.linspace(0.1, 2.0, 20)
nw_cv_mses = []

for h in bandwidths:
    fold_mses = []
    for train_idx, val_idx in kf.split(x):
        x_tr, x_val = x[train_idx], x[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        preds = nadaraya_watson(x_tr, y_tr, x_val, h)
        mse = np.mean((y_val - preds) ** 2)
        fold_mses.append(mse)
    nw_cv_mses.append(np.mean(fold_mses))

h_star = bandwidths[np.argmin(nw_cv_mses)]

# Plot MSE vs h
plt.figure()
plt.plot(bandwidths, nw_cv_mses, marker='o', color='blue')
plt.axvline(h_star, color='red', linestyle='--', label=f'Best h* = {h_star:.2f}')
plt.title('Nadaraya-Watson: CV-MSE vs Bandwidth h')
plt.xlabel('Bandwidth h')
plt.ylabel('MSE')
plt.legend()
plt.savefig('PLOT_1.jpg')
plt.close()

# --- 4-fold CV for Smoothing Splines ---
# In Python, UnivariateSpline uses 's' (smoothing factor) as lambda counterpart
lambdas = np.logspace(-2, 3, 20)
spline_cv_mses = []

for lam in lambdas:
    fold_mses = []
    for train_idx, val_idx in kf.split(x):
        x_tr, x_val = x[train_idx], x[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Sort training data because UnivariateSpline requires strictly increasing x
        sort_idx = np.argsort(x_tr)
        x_tr_s, y_tr_s = x_tr[sort_idx], y_tr[sort_idx]

        try:
            spline = UnivariateSpline(x_tr_s, y_tr_s, s=lam)
            preds = spline(x_val)
            mse = np.mean((y_val - preds) ** 2)
        except Exception:
            mse = np.mean((y_val - np.mean(y_tr_s)) ** 2) # Fallback if spline fails
        fold_mses.append(mse)
    spline_cv_mses.append(np.mean(fold_mses))

lam_star = lambdas[np.argmin(spline_cv_mses)]

# Plot MSE vs lambda
plt.figure()
plt.semilogx(lambdas, spline_cv_mses, marker='o', color='green')
plt.axvline(lam_star, color='red', linestyle='--', label=f'Best $\\lambda$* = {lam_star:.2f}')
plt.title('Smoothing Spline: CV-MSE vs $\\lambda$')
plt.xlabel('$\\lambda$ (Smoothing Parameter)')
plt.ylabel('MSE')
plt.legend()
plt.savefig('PLOT_2.jpg')
plt.close()

# --- Overlay Plot ---
fine_grid = np.linspace(min(x), max(x), 500)
fit_nw = nadaraya_watson(x, y, fine_grid, h_star)

sort_idx_all = np.argsort(x)
fit_spline = UnivariateSpline(x[sort_idx_all], y[sort_idx_all], s=lam_star)(fine_grid)

plt.figure(figsize=(10, 6))
plt.scatter(x, y, color='gray', alpha=0.6, label='Data points')
plt.plot(fine_grid, fit_nw, label=f'Nadaraya-Watson (h={h_star:.2f})', color='blue', lw=2)
plt.plot(fine_grid, fit_spline, label=f'Smoothing Spline ($\\lambda$={lam_star:.2f})', color='green', lw=2)
plt.title('Fitted Curves Overlay')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.savefig('PLOT_3.jpg')
plt.close()


# =====================================================================
# TASK 2: Artificial Data Generation (4 points)
# =====================================================================
print("--- Running Task 2 ---")

def generate_dataset(size):
    # X1 generation from 3-component Gaussian Mixture
    # Components setup: weights = [0.25, 0.25, 0.5]
    components = np.random.choice([0, 1, 2], size=size, p=[0.25, 0.25, 0.5])
    x1 = np.zeros(size)

    x1[components == 0] = np.random.normal(-1, 0.2, size=np.sum(components == 0))
    x1[components == 1] = np.random.normal(1, 0.2, size=np.sum(components == 1))
    x1[components == 2] = np.random.normal(5, 0.2, size=np.sum(components == 2))

    # X2 to X10 generation
    x_other = np.random.normal(0, 1, size=(size, 9))

    # Combine into 10-dimensional feature vector
    X = np.hstack((x1.reshape(-1, 1), x_other))

    # Generate binary labels Y based on P(Y=1|X)
    probs = np.where(x1 < 3, 0.1, 0.9)
    Y = np.random.binomial(1, probs)

    return X, Y

X_train, y_train = generate_dataset(1000)
X_test, y_test = generate_dataset(1000)


# =====================================================================
# TASK 3: Gradient Descent Implementation from Scratch (16 points)
# =====================================================================
print("--- Running Task 3 ---")

"""
TASK 3.1: DERIVATION AND GRADIENT COMMENT
The Risk Function given is:
R
"""
print("--- Running Task 3 ---")

"""
TASK 3.1: DERIVATION AND GRADIENT COMMENT
The Risk Function given is:
R(θ) = -1/n * ∑ [ ||x_i||_∞ * (y_i * x_i^T θ + log(1 - σ(x_i^T θ))) ] + 0.01 * ||θ||_2^2

Let's use the identity: 1 - σ(z) = 1 / (1 + e^z).
Thus, log(1 - σ(x_i^T θ)) = - log(1 + exp(x_i^T θ)).

The derivative with respect to θ of the loss component:
d/dθ [ y_i * x_i^T θ - log(1 + exp(x_i^T θ)) ]
= y_i * x_i - [ exp(x_i^T θ) / (1 + exp(x_i^T θ)) ] * x_i
= y_i * x_i - σ(x_i^T θ) * x_i
= (y_i - σ(x_i^T θ)) * x_i

Therefore, incorporating the negative sign and the weight ||x_i||_∞:
∇R(θ) = -1/n * ∑ [ ||x_i||_∞ * (y_i - σ(x_i^T θ)) * x_i ] + 0.02 * θ
      =  1/n * ∑ [ ||x_i||_∞ * (σ(x_i^T θ) - y_i) * x_i ] + 0.02 * θ
"""

def sigmoid(z):
    # Clip z to prevent numerical overflow in exp
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def compute_risk(X, y, theta):
    n_samples = X.shape[0]
    z = X @ theta
    sig = sigmoid(z)

    # Calculate L_infinity norm for each sample vector
    inf_norms = np.max(np.abs(X), axis=1)

    # Safe log calculation to avoid log(0)
    term1 = y * z
    term2 = np.log(1.0 - sig + 1e-15)

    inside_sum = inf_norms * (term1 + term2)
    risk = - (1.0 / n_samples) * np.sum(inside_sum) + 0.01 * np.sum(theta ** 2)
    return risk

def compute_accuracy(X, y, theta):
    z = X @ theta
    preds = (sigmoid(z) > 0.5).astype(int)
    return np.mean(preds == y)

# GD Parameters
lr = 0.01
epochs = 100
theta = np.zeros(X_train.shape[1]) # initialization at (0,...,0)

# Tracking history (including epoch 0 initialization baseline)
risk_history = [compute_risk(X_train, y_train, theta)]
train_acc_history = [compute_accuracy(X_train, y_train, theta)]
test_acc_history = [compute_accuracy(X_test, y_test, theta)]

n_samples = X_train.shape[0]
inf_norms = np.max(np.abs(X_train), axis=1)

# Gradient Descent Loop
for epoch in range(epochs):
    z = X_train @ theta
    sig = sigmoid(z)

    # Compute Gradient: 1/n * X^T @ (diag(inf_norms) @ (sig - y)) + 0.02 * theta
    errors = sig - y_train
    weighted_errors = inf_norms * errors
    grad = (1.0 / n_samples) * (X_train.T @ weighted_errors) + 0.02 * theta

    # Update weights
    theta = theta - lr * grad

    # Log metrics
    risk_history.append(compute_risk(X_train, y_train, theta))
    train_acc_history.append(compute_accuracy(X_train, y_train, theta))
    test_acc_history.append(compute_accuracy(X_test, y_test, theta))

# --- Plot Generation for Task 3 ---

# 3.3 Risk Evolution Plot
plt.figure()
plt.plot(range(epochs + 1), risk_history, color='purple', lw=2)
plt.title('Risk Function values over iterations')
plt.xlabel('Iteration / Epoch')
plt.ylabel('Risk R($\\theta$)')
plt.grid(True)
plt.savefig('RISK.pdf')
plt.close()

# 3.4 Train Accuracy Plot
plt.figure()
plt.plot(range(epochs + 1), train_acc_history, color='orange', lw=2)
plt.title('Training Accuracy over iterations')
plt.xlabel('Iteration / Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.savefig('TRAIN.pdf')
plt.close()

# 3.5 Test Accuracy Plot
plt.figure()
plt.plot(range(epochs + 1), test_acc_history, color='teal', lw=2)
plt.title('Testing Accuracy over iterations')
plt.xlabel('Iteration / Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.savefig('TEST.pdf')
plt.close()

--- Running Task 1 ---
weights.csv not found. Generating dummy data for Task 1 demonstration.
--- Running Task 2 ---
--- Running Task 3 ---
--- Running Task 3 ---


# FINAL TEST (Advanced Machine Learning) — Group B

**Date:** 2025-06-06

## Submission Instructions

- Upload files with solutions and plots to **Assignment on MS TEAMS**:
  - `test.ipynb` (or `.py` / `.r` with additional files `PLOT[plot_number].jpg`)
- **Deadline:** 9:50
- **Do NOT use archive formats** such as `.zip`.
- Uploading files after the deadline or using the wrong format will result in reduced points.

---

# Task 1 (10 points)

- Read the paired data \((x_i, y_i)\), \(i = 1, \ldots, n\), from `weights.csv` representing weights of some person (some dates are missing).
- Randomly split the \(n\) observations into four near equal-sized folds \(F_1, F_2, F_3, F_4\).

## 4-Fold Cross-Validation for Nadaraya–Watson Kernel Regression

- Choose a set of bandwidths \(\{h_1, \ldots, h_m\}\).
- For each \(h_j\), perform a 4-fold CV.
- Plot **MSE vs. \(h\)**.
- Select the bandwidth \(h^*\) that minimizes CV-MSE.

## 4-Fold Cross-Validation for Smoothing Splines Regression

- Choose a set of smoothing parameters \(\{\lambda_1, \ldots, \lambda_p\}\).
- For each \(\lambda_\ell\):
  - Fit the smoothing spline on three folds.
  - Compute MSE on the held-out fold.
  - Average over all four folds.
- Plot **MSE vs. \(\lambda\)**.
- Select the smoothing parameter \(\lambda^*\) that minimizes CV-MSE.

## Final Plot

- Plot the scatter of \((x_i, y_i)\).
- Overlay the fitted curves \(f_{h^*}(x)\) and \(f_{\lambda^*}(x)\) on a fine grid over the range of \(x\).

---

# Task 2 (4 points)

Generate artificial data as follows:

## Feature Generation

- Generate feature \(X_1\) from the mixture of 3 Gaussian distributions:

\[
0.25 \cdot N(-1,\; sd = 0.2)
+ 0.25 \cdot N(1,\; sd = 0.2)
+ 0.5 \cdot N(5,\; sd = 0.2)
\]

- Generate features \(X_2, \ldots, X_{10}\) from:

\[
N(0,\; sd = 1)
\]

## Class Variable

Generate a binary class variable such that:

\[
P(Y = 1 \mid X) =
\begin{cases}
0.1, & X_1 < 3 \\
0.9, & X_1 \ge 3
\end{cases}
\]

## Datasets

- Generate training data

\[
D_{train} = \{(x_i, y_i) : i = 1, \ldots, n\}
\]

with size \(n = 1000\), where \(x_i\) is a 10-dimensional feature vector and \(y_i\) is the class label.

- Generate testing data \(D_{test}\) of size \(n = 1000\) using the same scheme.

---

# Task 3 (16 points)

Assume that the model predicts

\[
P(Y = 1 \mid X = x)
\]

using the logistic sigmoid function

\[
\sigma(x^T \theta)
\]

and the optimal values of parameters \(\theta\) are found by minimizing the following weighted risk function on the training data:

\[
R(\theta)
=
-\frac{1}{n}
\sum_{i=1}^{n}
\left(
\|x_i\|_{\infty}
\, y_i \, x_i^T \theta
+
\log\bigl(1 - \sigma(x_i^T \theta)\bigr)
\right)
+
0.01 \|\theta\|_2^2
\]

To make a prediction for a new observation, apply the rule:

\[
\hat{y} = 1
\quad \text{when} \quad
\sigma(x^T \hat{\theta}) > 0.5
\]

## Requirements

### 1. Gradient Calculation

- Calculate the gradient \(\nabla R(\theta)\) of the risk function.
- Write the solution in a comment.

### 2. Gradient Descent

- Implement **from scratch** gradient descent (GD).
- Use:
  - Learning rate: `lr = 0.01`
  - Number of epochs: `100`
- Initialize:

\[
\theta = (0, \ldots, 0)
\]

### 3. Risk Plot

- Generate a plot showing how the value of the risk changes over iterations.
- Save the plot as:

```text
RISK.pdf
```

### 4. Training Accuracy Plot

- Generate a plot showing how the training accuracy changes over iterations.
- Include the prediction for the initialization point.
- Save the plot as:

```text
TRAIN.pdf
```

### 5. Testing Accuracy Plot

- Generate a plot showing how the testing accuracy changes over iterations.
- Include the prediction for the initialization point.
- Save the plot as:

```text
TEST.pdf
```